# Module 05 — Lecture 1: Synaptic Models on GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/GPU-Programming-For-Computational-Neuroscience/blob/main/module_05_networks_plasticity/01_synaptic_models.ipynb)

---

Synaptic connections transmit information between neurons. Each synapse has:
1. A **weight** w (synaptic strength)
2. A **dynamic** — the conductance evolves over time after a spike

**Learning objectives:**
- Implement exponential and alpha-function synaptic conductances on GPU
- Model excitatory (AMPA/NMDA) and inhibitory (GABA) synapses
- Understand current-based vs conductance-based synaptic models
- Add synaptic input to the LIF simulation from Module 03

In [ ]:
!nvidia-smi

## 1. Synaptic Models

### Current-Based (simple)
$$I_{syn} = w \cdot \delta(t - t_{spike}) \text{ (instant pulse)}$$
$$I_{syn}(t) = w \cdot e^{-(t-t_{spike})/\tau_s} \text{ (exponential decay)}$$

Simple, but ignores the driving force — synapse doesn't become weaker when V approaches reversal potential.

### Conductance-Based (more realistic)
$$\tau_s \frac{dg}{dt} = -g, \quad g \to g + w \text{ on pre-spike}$$
$$I_{syn} = g(t) \cdot (V - E_{syn})$$

The driving force $(V - E_{syn})$ makes excitatory synapses push V toward $E_E = 0$ mV and inhibitory synapses push toward $E_I = -80$ mV.

### Alpha Function
$$g(t) = \frac{w \cdot t}{\tau_s} e^{1 - t/\tau_s}$$
Rises and then falls — models the delay between pre-synaptic release and peak conductance.

| Model | τ (ms) | E_syn (mV) | Biological target |
|-------|---------|------------|-------------------|
| AMPA (E) | 5 | 0 | Fast excitation |
| NMDA (E) | 100 | 0 | Slow excitation (coincidence detection) |
| GABA-A (I) | 10 | -70 to -80 | Fast inhibition |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dt = 0.1
T = int(100 / dt)
t = np.arange(T) * dt

# Simulate synaptic conductances after a spike at t=10 ms
t_spike = 10.0

# Exponential (E)
tau_E = 5.0
g_exp = np.zeros(T)
for i in range(1, T):
    g_exp[i] = g_exp[i-1] * np.exp(-dt / tau_E)
    if abs(t[i] - t_spike) < dt / 2:
        g_exp[i] += 1.0  # weight = 1

# Alpha function
t_shifted = np.maximum(t - t_spike, 0)
g_alpha = t_shifted / tau_E * np.exp(1 - t_shifted / tau_E)
g_alpha[t < t_spike] = 0

# NMDA (slow)
tau_N = 80.0
g_nmda = np.zeros(T)
for i in range(1, T):
    g_nmda[i] = g_nmda[i-1] * np.exp(-dt / tau_N)
    if abs(t[i] - t_spike) < dt / 2:
        g_nmda[i] += 1.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(t, g_exp, 'b', lw=2, label=f'AMPA exponential (τ={tau_E} ms)')
ax1.plot(t, g_alpha, 'r--', lw=2, label=f'Alpha function (τ={tau_E} ms)')
ax1.plot(t, g_nmda, 'g', lw=2, label=f'NMDA exponential (τ={tau_N} ms)')
ax1.axvline(t_spike, color='k', linestyle=':', alpha=0.5, label='Pre-spike')
ax1.set_xlabel('Time (ms)', fontsize=12)
ax1.set_ylabel('Conductance g (normalised)', fontsize=12)
ax1.set_title('Synaptic Conductance Models', fontsize=13)
ax1.legend(fontsize=10); ax1.grid(True, alpha=0.3)

# Conductance-based current: I = g * (V - E_syn)
V_values = np.array([-70, -60, -55, -40, 0, 20])
E_E, E_I = 0, -80
t_peak = t[np.argmax(g_exp)]
g_peak = g_exp.max()

for V in V_values:
    I_E = g_peak * (V - E_E)
    I_I = g_peak * (V - E_I)
    ax2.scatter([V], [-I_E], s=80, color='b', marker='o')
    ax2.scatter([V], [-I_I], s=80, color='r', marker='s')

V_cont = np.linspace(-90, 30, 200)
ax2.plot(V_cont, [-g_peak*(v-E_E) for v in V_cont], 'b', lw=2, label='Excitatory (E_E=0 mV)')
ax2.plot(V_cont, [-g_peak*(v-E_I) for v in V_cont], 'r', lw=2, label='Inhibitory (E_I=-80 mV)')
ax2.axhline(0, color='k', lw=0.5)
ax2.axvline(-65, color='gray', linestyle=':', alpha=0.5, label='V_rest')
ax2.set_xlabel('Membrane Voltage V (mV)', fontsize=12)
ax2.set_ylabel('Synaptic current I_syn = -g·(V-E) (inward+)', fontsize=11)
ax2.set_title('Conductance-Based Driving Force', fontsize=13)
ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('synaptic_models.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# GPU implementation of conductance-based LIF network
%%writefile syn_lif.cu
#include <stdio.h>
#include <cuda_runtime.h>

__constant__ float c_dt, c_tau_m, c_E_L, c_Rm, c_V_th, c_V_reset;
__constant__ float c_tau_E, c_tau_I, c_E_E, c_E_I;
__constant__ int c_T_ref;

// Paired neuron kernel: voltage + conductances
__global__ void update_cond_lif(
    float* V, float* g_E, float* g_I, int* ref,
    const float* I_ext, int* fired, int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N) return;

    // Decay conductances (exact exponential)
    g_E[i] *= expf(-c_dt / c_tau_E);
    g_I[i] *= expf(-c_dt / c_tau_I);

    fired[i] = 0;
    if (ref[i] > 0) { ref[i]--; V[i] = c_V_reset; return; }

    // Total synaptic current (inward = positive)
    float I_syn = -(g_E[i] * (V[i] - c_E_E) + g_I[i] * (V[i] - c_E_I));
    float dV = c_dt / c_tau_m * (-(V[i]-c_E_L) + c_Rm*(I_ext[i] + I_syn));
    V[i] += dV;

    if (V[i] >= c_V_th) {
        V[i] = c_V_reset; ref[i] = c_T_ref; fired[i] = 1;
    }
}

// Spike propagation: for each neuron that fired,
// scatter its weight into the post-synaptic conductance
__global__ void propagate(const int* fired, const int* rowptr, const int* col,
                           const float* w, const int* type,
                           float* g_E, float* g_I, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= N || !fired[i]) return;
    for (int k = rowptr[i]; k < rowptr[i+1]; k++) {
        int j = col[k];
        if (type[k] == 0) atomicAdd(&g_E[j], w[k]);
        else              atomicAdd(&g_I[j], w[k]);
    }
}

int main() {
    printf("Conductance-based LIF kernel compiled successfully.\n");
    printf("See network_sim.cu for the full simulation.\n");
    return 0;
}

In [ ]:
!nvcc -O2 -o syn_lif syn_lif.cu && ./syn_lif

## Summary

| Model | Equation | Advantage | When to use |
|-------|----------|-----------|-------------|
| Current-based | I = w × δ(t) | Simplest | Large-scale speed |
| Exponential g | dg/dt = -g/τ | Smooth, accurate | Most simulations |
| Alpha function | g ∝ t·e^(-t/τ) | Captures rise time | When timing matters |
| Conductance-based | I = g·(V - E_syn) | Driving force | Biophysical realism |

**Next lecture:** Sparse connectivity representation (CSR format) for efficient spike propagation.